In [1]:
!pip install -q langchain langchain-community langchain-anthropic \
    sentence-transformers faiss-cpu beautifulsoup4 \
    requests anthropic python-dotenv tiktoken

In [2]:
pip install -U langchain langchain-core langchain-community langchain-text-splitters langchain-anthropic

  Using cached langchain_core-1.2.30-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_anthropic-1.4.0-py3-none-any.whl.metadata (3.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 2.2 MB/s eta 0:00:00
Using cached langchain_core-1.2.30-py3-none-any.whl (513 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached langchain_text_splitters-1.1.1-py3-none-any.whl (35 kB)
Using cached langchain_anthropic-1.4.0-py3-none-any.whl (48 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.3/378.3 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.1.147
    Uninstalling langsmith-0.1.147:
      Successfully uninstalled langsmith-0.1.147
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.2.43
    Uninstal

## Imports

In [3]:
pip install langchain-groq

  Using cached langchain_core-0.2.43-py3-none-any.whl.metadata (6.2 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
Using cached langchain_core-0.2.43-py3-none-any.whl (397 kB)
Using cached langsmith-0.1.147-py3-none-any.whl (311 kB)
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.7.32
    Uninstalling langsmith-0.7.32:
      Successfully uninstalled langsmith-0.7.32
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.30
    Uninstalling langchain-core-1.2.30:
      Successfully uninstalled langchain-core-1.2.30
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-text-splitters 1.1.1 requires langchain-core<2.0.0,>=1.2.13, but you have langchain-core 0.2.43 which is incompatible.
langchain-anthropic 1.4.0 requires langchain-core<2.0.0,>=1.2.19, but you have langch

In [4]:
!pip install -U \
langchain==0.2.17 \
langchain-core==0.2.43 \
langchain-community==0.2.17 \
langchain-text-splitters==0.2.4 \
langchain-groq==0.1.9 \
faiss-cpu \
sentence-transformers

  Using cached langchain-0.2.17-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_community-0.2.17-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl.metadata (2.3 kB)
Using cached langchain-0.2.17-py3-none-any.whl (1.0 MB)
Using cached langchain_community-0.2.17-py3-none-any.whl (2.3 MB)
Using cached langchain_text_splitters-0.2.4-py3-none-any.whl (25 kB)
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 1.1.1
    Uninstalling langchain-text-splitters-1.1.1:
      Successfully uninstalled langchain-text-splitters-1.1.1
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.15
    Uninstalling langchain-1.2.15:
      Successfully uninstalled langchain-1.2.15
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.4.1
    Uninstalling langchain-community-0.4.1:
      Successfully uninstalled langchain-commu

In [5]:
import os
import json
import re
import requests
import numpy as np
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# LangChain core
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.history_aware_retriever import create_history_aware_retriever
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage
from IPython.display import display

import warnings
warnings.filterwarnings('ignore')

In [6]:
print(os.listdir())

['.config', '.env.txt', 'sample_data']


##  API Key Setup

In [7]:
load_dotenv(".env.txt")
groq_api_key = os.getenv("GROQ_API_KEY")
print(groq_api_key)

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


## Scrape Finance Data

In [8]:
from logging import exception
dataset_folder = "./finance_dataset"
os.makedirs(dataset_folder, exist_ok=True)

urls = {
    "Stock_Scam": "https://www.investor.gov/introduction-investing/general-resources/news-alerts/alerts-bulletins/investor-bulletins/social-media-stock-scams",
    "Investor_Bulletin": "https://www.investor.gov/introduction-investing/general-resources/news-alerts/alerts-bulletins/investor-bulletins/investorgov-tips-2026-investor-bulletin",
    "Crypto-Asset": "https://www.investor.gov/introduction-investing/general-resources/news-alerts/alerts-bulletins/investor-bulletins/crypto-asset-custody-basics-retail-investors-investor-bulletin-0",
    "Tips-2025": "https://www.investor.gov/introduction-investing/general-resources/news-alerts/alerts-bulletins/investor-bulletins/ten-investment-tips-2025-investor-bulletin",
    "Updated": "https://www.investor.gov/introduction-investing/general-resources/news-alerts/alerts-bulletins/investor-bulletins/updated",
    "Crypto-Scams": "https://www.investor.gov/introduction-investing/general-resources/news-alerts/alerts-bulletins/investor-alerts/crypto-scams",
    "AI-Fraud": "https://www.investor.gov/introduction-investing/general-resources/news-alerts/alerts-bulletins/investor-alerts/artificial-intelligence-fraud",
    "Relationship-Scam": "https://www.investor.gov/introduction-investing/general-resources/news-alerts/alerts-bulletins/investor-alerts/relationship-investment-scams-investor-alert",
    "Protect-TSP": "https://www.investor.gov/introduction-investing/general-resources/news-alerts/alerts-bulletins/investor-alerts/protect-your-tsp-account"
}

headers = {"User-Agent": "Mozilla/5.0"}

for name, url in urls.items():
    print(f"\nFetching {name} ...")
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        # Remove nav, footer, script noise
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        paragraphs = soup.find_all("p")
        text = "\n\n".join(p.get_text().strip() for p in paragraphs if len(p.get_text().strip()) > 40)

        with open(f"{dataset_folder}/{name}.txt", "w", encoding="utf-8") as f:
            f.write(text)

        print(f"{name}.txt saved ({len(text)} chars)")
    except Exception as e:
        print(f"Failed {name}: {e}")


Fetching Stock_Scam ...
Stock_Scam.txt saved (6069 chars)

Fetching Investor_Bulletin ...
Investor_Bulletin.txt saved (3149 chars)

Fetching Crypto-Asset ...
Crypto-Asset.txt saved (5093 chars)

Fetching Tips-2025 ...
Tips-2025.txt saved (7075 chars)

Fetching Updated ...
Updated.txt saved (4231 chars)

Fetching Crypto-Scams ...
Crypto-Scams.txt saved (9092 chars)

Fetching AI-Fraud ...
AI-Fraud.txt saved (6960 chars)

Fetching Relationship-Scam ...
Relationship-Scam.txt saved (9827 chars)

Fetching Protect-TSP ...
Protect-TSP.txt saved (5871 chars)


## Clean, Chunk with LangChain Splitter & Build Documents

In [9]:
def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', text)          # collapse whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)    # max 2 newlines
    text = re.sub(r'[^\x00-\x7F]+', '', text) # remove non-ASCII
    return text.strip()

# LangChain's smarter splitter — respects sentence/paragraph boundaries
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,          # characters per chunk (more precise than word count)
    chunk_overlap=50,        # overlap so context isn't lost at boundaries
    separators=["\n\n", "\n", ". ", " ", ""]
)

all_documents = [] # LangChain Document objects

for filename in os.listdir(dataset_folder):
    if not filename.endswith(".txt"):
        continue

    filepath = os.path.join(dataset_folder, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        raw_text = f.read()

    cleaned = clean_text(raw_text)
    chunks = splitter.split_text(cleaned)

    for i, chunk in enumerate(chunks):
      doc = Document(
          page_content=chunk,
          metadata={
              "source": filename.replace(".txt", ""),
              "chunk_id": f"{filename}_chunk_{i}",
              "topic": filename.replace(".txt", "").replace("_", " ").title()
          }
      )
      all_documents.append(doc)

print(f"Total LangChain documents created: {len(all_documents)}")
print(f"Sample Documents:")
print(all_documents[0].page_content)
print(all_documents[0].metadata)

Total LangChain documents created: 142
Sample Documents:
The .gov means its official. Federal government websites often end in .gov or .mil. Before sharing sensitive information, make sure youre on a federal government site. The site is secure. The https:// ensures that you are connecting to the official website and that any information you provide is encrypted and transmitted securely
{'source': 'Stock_Scam', 'chunk_id': 'Stock_Scam.txt_chunk_0', 'topic': 'Stock Scam'}


## Build FAISS Vector Store via LangChain

In [10]:
print("Loading embedding model...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2", # embedding model to convert text to number (vector)
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True} # normalise for better cosine similarity
)

print("Building FAISS vector store...")
vectorstore = FAISS.from_documents(all_documents, embedding_model)

# Save to disk so no need to re-embed every time
vectorstore.save_local("./finance_faiss_index")
print("FAISS vector store built and saved!")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building FAISS vector store...
FAISS vector store built and saved!


##  LangChain Retriever with MMR

In [22]:
# MMR = Maximal Marginal Relevance — balances relevance + diversity
# avoids returning 3 chunks that all say the same thing
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5, # number of chunks
        "fetch_k": 20, # number of chunks to fetch
        "lambda_mult": 0.7 # 0 = max diversity, 1 = max relevance
    }
)

# Quick Test
test_docs = retriever.get_relevant_documents("What is the scam?")
print(f"Retrieved {len(test_docs)} documents:")
for doc in test_docs:
    print(f"  - [{doc.metadata['topic']}] {doc.page_content[:200]}...")

Retrieved 9 documents:
  - [Relationship-Scam] . These scams sometimes are referred to as long cons, meaning theres a long, slow build before the fraudster springs their trap. Once they find a target whos willing to engage with them, these fraudst...
  - [Stock Scam] . stock exchanges) and to send screenshots of your purchases to the adviser. The next thing you know, the stock price plummets and you lose most of your investment. SEC staff created the following moc...
  - [Tips-2025] . Whether you are a first-time investor or have been investing for years, here are 10 tips from the SECs Office of Investor Education and Assistance to help you make better informed investment decisio...
  - [Crypto-Scams] . One way this type of scam can play out is that after a fraudster has established an online relationship with you, the fraudster may claim to know about lucrative investment or trading opportunities,...
  - [Ai-Fraud] . These bad actors might use catchy AI-related buzzwords and make clai

## Prompt Engineering

In [23]:
SYSTEM_PROMPT = """You are FinanceBot 💼, an intelligent financial education assistant backed by verified sources including the SEC (U.S. Securities and Exchange Commission) and Investopedia.

---

## WHO YOU ARE
You are a knowledgeable, friendly financial educator. You explain complex financial concepts in a clear, structured, and engaging way — like a smart friend who happens to be a financial expert.

---

## HOW YOU RESPOND

**Structure your answers like this:**

1. **Direct Answer** — Start with a clear 1-2 sentence answer to the question
2. **Explanation** — Break it down with context and reasoning
3. **Key Points** — Use bullet points (•) for lists, comparisons, or steps
4. **Example** (if helpful) — Give a simple real-world example using Malaysian Ringgit (RM) where relevant
5. **Source** — Mention which source the information comes from

**Formatting rules:**
- Use **bold** for important terms and concepts
- Use bullet points (•) for lists of 3 or more items
- Use numbered lists for steps or sequences
- Use `code-style` for formulas (e.g. `Return = (Profit / Investment) × 100%`)
- Keep paragraphs short — max 3 sentences each
- Add a blank line between sections for readability

---

## YOUR BOUNDARIES

You CAN:
- Explain financial concepts, terms, and strategies
- Compare financial products or approaches
- Help users understand market mechanics
- Provide general investment education

You CANNOT:
- Give personalised investment advice
- Predict market movements
- Recommend specific stocks, funds, or brokers
- Act as a licensed financial advisor

---

## WHEN CONTEXT IS INSUFFICIENT
If the retrieved context does not contain enough information to answer confidently, respond with:
"I don't have enough verified information on that topic in my current knowledge base. I recommend checking **Investopedia.com** or **investor.gov** directly for accurate guidance."

---

## TONE & STYLE
- Friendly but professional — like a knowledgeable mentor
- Encouraging, never condescending
- Concise — avoid unnecessary filler words
- Use "you" and "your" to keep it personal

---

## RETRIEVED CONTEXT
Use ONLY the following verified content to ground your answer:
{context}

---

⚠️ **Disclaimer:** Always end every response with this exact line:
"*This information is for educational purposes only and does not constitute professional financial advice. Please consult a certified financial advisor (CFP) for personalised guidance.*"
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

print("Prompt template created!")

Prompt template created!


## Initialize LLM + Conversational Memory

In [24]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key=groq_api_key,
    model="llama-3.3-70b-versatile",
    temperature=0.3,  # low = more factual, high = more creative
    max_tokens=1024,
)

# Chat history is managed manually in ask_financebot()
# using HumanMessage / AIMessage objects — no memory object needed
chat_history = []
MAX_HISTORY = 5

print("LLM initialized!")

LLM initialized!


## Build the Full LangChain RAG Chain

In [25]:
# Step 1 — chain that fills {context} into prompt and calls LLM
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

# Step 2 — full RAG chain (retriever feeds context into combine_docs_chain)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

print("RAG chain ready!")

RAG chain ready!


## Chatbot Query Function with Source Attribution

In [26]:
# Manual history — always a clean list, never a string
chat_history = []
MAX_HISTORY = 5

def ask_financebot(question: str, verbose: bool = True) -> dict:
    global chat_history

    result = rag_chain.invoke({
        "input": question,           # matches ("human", "{input}") in your prompt
        "chat_history": chat_history # always a list of message objects
    })

    answer = result["answer"]
    sources = list({doc.metadata["topic"] for doc in result.get("context", [])})

    # Save to history as proper message objects
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

    # Keep only last 5 exchanges
    if len(chat_history) > MAX_HISTORY * 2:
        chat_history = chat_history[-(MAX_HISTORY * 2):]

    if verbose:
        print(f"\n{'='*60}")
        print(f"❓ Question: {question}")
        print(f"{'='*60}")
        print(f"\n🤖 FinanceBot:\n{answer}")
        print(f"\n📚 Sources used: {', '.join(sources)}")
        print(f"{'='*60}\n")

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "source_documents": result.get("context", [])
    }

def reset_chat():
    global chat_history
    chat_history = []
    print("Chat history cleared!")

## Multi-turn Conversation Test

In [27]:
reset_chat()

conversation = [
    "What is a crypto scam and how does it work?",
    "How is that different from AI fraud?",
    "What are the warning signs I should watch out for?",
    "If I think I've been scammed, what should I do?",
]

print("Starting multi-turn conversation...\n")
for question in conversation:
    ask_financebot(question)

Chat history cleared!
🗣️ Starting multi-turn conversation...


❓ Question: What is a crypto scam and how does it work?

🤖 FinanceBot:
A **crypto scam** is a type of investment scam that involves fraudulent activities related to **crypto assets**, such as cryptocurrencies, tokens, or coins. These scams often involve fraudsters who use fake online relationships, promises of lucrative investment opportunities, or fake investment platforms to trick victims into sending them money or crypto assets.

The scam typically works by the fraudster establishing an online relationship with the victim, gaining their trust, and then introducing a fake investment opportunity, such as an **Initial Coin Offering (ICO)** or a crypto asset trading platform. The fraudster may claim to have inside information or promise unusually high returns to convince the victim to invest. In reality, the victim is simply sending money or crypto assets directly to the fraudster's wallet or account.

**Key Points:**
• Frau

## Evaluation

In [31]:
reset_chat()

eval_questions = [
    "What is a stock scam and how do fraudsters manipulate prices?",
    "What are the risks of crypto asset custody for retail investors?",
    "How do relationship investment scams work?",
    "What are the top 10 investment tips for 2025?",
    "How does AI get used in financial fraud?",
]

eval_results = []

for q in eval_questions:
    result = ask_financebot(q, verbose=False)

    answer_length = len(result["answer"].split())
    has_disclaimer = "educational purposes" in result["answer"].lower()
    num_sources = len(result["sources"])

    # Check if retrieved sources are relevant to the question
    expected_keywords = {
        "What are the top 10 investment tips for 2025?":                    "Tips-2025",
        "What is a stock scam and how do fraudsters manipulate prices?":     "Stock_Scam",
        "What are the risks of crypto asset custody for retail investors?":  "Crypto-Asset",
        "How do relationship investment scams work?":                        "Relationship-Scam",
        "How does AI get used in financial fraud?":                          "AI-Fraud",
    }
    expected_source = expected_keywords.get(q, "")
    source_hit = "✅" if any(expected_source.lower() in s.lower() for s in result["sources"]) else "❌"

    eval_results.append({
        "Question":       q[:55] + "..." if len(q) > 55 else q,
        "Answer Words":   answer_length,
        "Has Disclaimer": "✅" if has_disclaimer else "❌",
        "Source Hit":     source_hit,
        "Sources Used":   num_sources,
        "Sources":        ", ".join(result["sources"]),
    })

df = pd.DataFrame(eval_results)
display(df)

Chat history cleared!


,Question,Answer Words,Has Disclaimer,Source Hit,Sources Used,Sources
0,What is a stock scam and how do fraudsters man...,234,✅,❌,4,"Stock Scam, Ai-Fraud, Relationship-Scam, Crypt..."
1,What are the risks of crypto asset custody for...,275,✅,✅,3,"Crypto-Scams, Ai-Fraud, Crypto-Asset"
2,How do relationship investment scams work?,343,✅,✅,5,"Crypto-Scams, Ai-Fraud, Tips-2025, Relationshi..."
3,What are the top 10 investment tips for 2025?,387,✅,✅,5,"Stock Scam, Crypto-Scams, Updated, Ai-Fraud, T..."
4,How does AI get used in financial fraud?,313,✅,✅,3,"Crypto-Scams, Ai-Fraud, Relationship-Scam"
